# Model 4 — Recurrent family (LSTM / BiLSTM / DeepConvLSTM)

**Owner:** Nandan Reddy Maram (230911032)
**Papers covered:** [3] Ordóñez & Roggen (2016), [4] Hammerla et al. (2016)
**Literature-review alignment:** [3] → `deepconvlstm`, [4] → `lstm` / `bilstm` family comparison

Three recurrent architectures, all trained on the **same** processed data
(`sequence.npz`, shape `N x 128 x 9`), the **same** subject-independent split, the
**same** optimiser and the **same** early-stopping rule, so their rows in
`results/comparison_table.csv` are directly comparable with the MLP, CNN and
Transformer rows.

| run | architecture | reference |
|---|---|---|
| `lstm` | 2 x LSTM(128), dropout 0.5, mean+max pooling | [4] Hammerla et al. 2016 |
| `bilstm` | 2 x BiLSTM(90 per direction), parameter-matched to `lstm` | [3] Ordóñez & Roggen 2016 |
| `deepconvlstm` | 5 x Conv1D(64, k=5) + BN + ReLU → 2 x LSTM(128) | [3] Ordóñez & Roggen 2016 |

The model code and the whole training / evaluation loop live in `rnn_models.py` and
`train_rnn.py` next to this notebook, so the numbers in the report are produced by the
exact code that is committed. Run all three with `!python train_rnn.py`, or step through
this notebook to run them one at a time.

In [ ]:
import os, sys, json, time, copy
import numpy as np
import torch, torch.nn as nn

PROJECT_DIR = os.environ.get("HAR_PROJECT_DIR", "/content/drive/MyDrive/DL Proj")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

HERE = os.path.dirname(os.path.abspath("train_rnn.py")) if os.path.exists("train_rnn.py") else "."
PROC_DIR = os.path.join(PROJECT_DIR, "processed")
assert os.path.isfile(os.path.join(PROC_DIR, "har_common.py")), "Run HAR_Preprocessing.ipynb first."
sys.path.insert(0, PROC_DIR)
sys.path.insert(0, HERE)
import har_common as hc
import rnn_models
from train_rnn import HP, run_one, SEED, DEVICE, USE_AMP

print(f"Device: {DEVICE} {torch.cuda.get_device_name(0) if USE_AMP else ''} | torch {torch.__version__}")
print(f"Dataset version: {hc.DATASET_VERSION} | classes: {hc.CLASSES}")
print(f"Owner: {rnn_models.OWNER}")
print(f"Shared training protocol: {HP}")

In [ ]:
data = hc.load_data("sequence", augmented_train=False)
for sp in ("train", "val", "test"):
    print(f"{sp:5s} X {data[f'X_{sp}'].shape}  y {np.bincount(data[f'y_{sp}'], minlength=6)}")

SEQ_LEN, N_CH = data["X_train"].shape[1:]
for name, (cls, hp) in rnn_models.MODELS.items():
    net, _ = rnn_models.build(name, N_CH, SEQ_LEN, hc.N_CLASSES)
    print(f"{name:14s} {type(net).__name__:13s} params {sum(p.numel() for p in net.parameters()):>9,}  hp {hp}")

In [ ]:
# ------------------------------------------------------------------ LSTM
RECORDS = {}
RECORDS["lstm"] = run_one("lstm", data)

In [ ]:
# --------------------------------------------------------------- BiLSTM
RECORDS["bilstm"] = run_one("bilstm", data)

In [ ]:
# -------------------------------------------------------- DeepConvLSTM
RECORDS["deepconvlstm"] = run_one("deepconvlstm", data)

In [ ]:
import pandas as pd

table = pd.read_csv(os.path.join(hc.RESULTS_DIR, "comparison_table.csv"))
cols = ["model_name", "run_tag", "owner", "test_accuracy", "test_macro_f1", "test_cohen_kappa",
        "f1_SITTING", "f1_STANDING", "sit_stand_confusion_rate", "walking_variants_confusion_rate",
        "subject_acc_mean", "subject_acc_min", "params_total", "model_size_mb",
        "flops_per_window", "latency_ms_cpu_b1", "train_time_s", "epochs_trained", "best_epoch"]
table[table.model_name.isin(["lstm", "bilstm", "deepconvlstm"])][cols].round(4)

## Notes

* Model selection is on **validation macro-F1**; the test set is used once, at the end.
* Seed 42, AdamW `lr=1e-3`, `weight_decay=0.01`, batch 64, gradient clipping 1.0, patience 20
  — identical to the CNN and Transformer baselines, so only the architecture differs.
* `flops_per_window` for these models is an analytic GEMM count (`2 x MACs`) because
  `torch.utils.flop_counter` does not trace the fused ATen LSTM kernels. The CNN and
  Transformer rows use the counter directly; both are GEMM counts under the same convention.
* `latency_ms_cpu_b1` was measured on the machine that produced these rows and is **not**
  comparable with the Tesla T4 measurements of the CNN/Transformer rows unless re-measured on
  the same host. This is stated in each row's `notes`.